# Menu Recommender System - Scala (SparkML)

This notebook uses SparkML Linear Regression to recommend menus for each day of the week for people with different allergen restrictions:
- Eggs allergy
- Gluten allergy  
- Lactose intolerance
- Nuts allergy
- Shellfish allergy

In [1]:
// Load Spark dependencies for Scala 2.12.12
// Using Spark 3.5.0 which is compatible with Scala 2.12.12
import $ivy.`org.apache.spark::spark-sql:3.5.0`
import $ivy.`org.apache.spark::spark-mllib:3.5.0`
import $ivy.`org.apache.spark::spark-core:3.5.0`

println("Spark dependencies loaded successfully!")

Spark dependencies loaded successfully!


import $ivy.$                                  

import $ivy.$                                    

import $ivy.$                                   



In [2]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.ml.regression.LinearRegression
import org.apache.spark.ml.feature.{VectorAssembler, StringIndexer, OneHotEncoder}
import org.apache.spark.ml.Pipeline
import org.apache.spark.sql.functions._
import org.apache.spark.sql.types._

// Create Spark Session with increased memory for ALS training
val spark = SparkSession.builder()
    .appName("MenuRecommenderScala")
    .master("local[*]")  // Run Spark in local mode with all available cores
    .config("spark.sql.shuffle.partitions", "200")  // Increased for better parallelism
    .config("spark.driver.memory", "8g")  // Increased from 4g to 8g
    .config("spark.executor.memory", "8g")  // Increased from 4g to 8g
    .config("spark.driver.maxResultSize", "4g")  // Allow larger result sets
    .config("spark.memory.fraction", "0.8")  // Use 80% of heap for Spark execution
    .config("spark.memory.storageFraction", "0.3")  // 30% for storage, 70% for execution
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")  // Disable Arrow for Scala
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")  // Use Kryo for better performance
    .config("spark.checkpoint.dir", "checkpoint")  // Enable checkpointing for memory management
    .config("spark.sql.adaptive.enabled", "true")  // Enable adaptive query execution
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")  // Coalesce partitions adaptively
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
spark.sparkContext.setCheckpointDir("checkpoint")  // Set checkpoint directory

println("Spark Session created successfully!")
println("IMPORTANT: If you see OutOfMemoryError, restart the kernel and re-run from this cell!")

SLF4J: No SLF4J providers were found.
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See https://www.slf4j.org/codes.html#noProviders for further details.


Spark Session created successfully!
IMPORTANT: If you see OutOfMemoryError, restart the kernel and re-run from this cell!


import org.apache.spark.sql.SparkSession

import org.apache.spark.ml.regression.LinearRegression

import org.apache.spark.ml.feature.{VectorAssembler, StringIndexer, OneHotEncoder}

import org.apache.spark.ml.Pipeline

import org.apache.spark.sql.functions._

import org.apache.spark.sql.types._

// Create Spark Session with increased memory for ALS training

spark: SparkSession = org.apache.spark.sql.SparkSession@3e857df

In [3]:
// Load restaurant menus data
val basePath = "output/restaurant_menus"

// Load all restaurant menus
val allMenusDF = spark.read.parquet(s"$basePath/all_restaurant_menus.parquet")

// Load allergen-specific menus
val eggAllergyMenus = spark.read.parquet(s"$basePath/egg_allergy_restaurant_menu.parquet")
val glutenFreeMenus = spark.read.parquet(s"$basePath/gluten_free_restaurant_menu.parquet")
val lactoseIntolerantMenus = spark.read.parquet(s"$basePath/lactose_intolerant_restaurant_menu.parquet")
val nutAllergyMenus = spark.read.parquet(s"$basePath/nut_allergy_restaurant_menu.parquet")
val shellfishAllergyMenus = spark.read.parquet(s"$basePath/shellfish_allergy_restaurant_menu.parquet")

println("Data loaded successfully!")
println(s"Total menus: ${allMenusDF.count()}")
println(s"Egg allergy menus: ${eggAllergyMenus.count()}")
println(s"Gluten-free menus: ${glutenFreeMenus.count()}")
println(s"Lactose intolerant menus: ${lactoseIntolerantMenus.count()}")
println(s"Nut allergy menus: ${nutAllergyMenus.count()}")
println(s"Shellfish allergy menus: ${shellfishAllergyMenus.count()}")

// Show schema
allMenusDF.printSchema()
allMenusDF.show(5, truncate=false)

Data loaded successfully!
Total menus: 99
Egg allergy menus: 20
Gluten-free menus: 20
Lactose intolerant menus: 19
Nut allergy menus: 20
Shellfish allergy menus: 20
root
 |-- meal_id: long (nullable = true)
 |-- starter_title: string (nullable = true)
 |-- starter_ingredients: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- starter_directions: string (nullable = true)
 |-- starter_link: string (nullable = true)
 |-- main_title: string (nullable = true)
 |-- main_ingredients: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- main_directions: string (nullable = true)
 |-- main_link: string (nullable = true)
 |-- dessert_title: string (nullable = true)
 |-- dessert_ingredients: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- dessert_directions: string (nullable = true)
 |-- dessert_link: string (nullable = true)
 |-- dietary_restriction: string (nullable = true)
 |-- restriction_description: string (nulla

basePath: String = "output/restaurant_menus"
allMenusDF: org.apache.spark.sql.package.DataFrame = [meal_id: bigint, starter_title: string ... 13 more fields]
eggAllergyMenus: org.apache.spark.sql.package.DataFrame = [meal_id: bigint, starter_title: string ... 13 more fields]
glutenFreeMenus: org.apache.spark.sql.package.DataFrame = [meal_id: bigint, starter_title: string ... 13 more fields]
lactoseIntolerantMenus: org.apache.spark.sql.package.DataFrame = [meal_id: bigint, starter_title: string ... 13 more fields]
nutAllergyMenus: org.apache.spark.sql.package.DataFrame = [meal_id: bigint, starter_title: string ... 13 more fields]
shellfishAllergyMenus: org.apache.spark.sql.package.DataFrame = [meal_id: bigint, starter_title: string ... 13 more fields]

In [4]:
// Load additional datasets for feature engineering
val nutritionalProfilesDF = spark.read.parquet("output/nutritional_profiles/nutritional_profiles.parquet")
val recipesDF = spark.read.parquet("data/recipes_data.parquet")

println("Additional datasets loaded!")
nutritionalProfilesDF.printSchema()

Additional datasets loaded!
root
 |-- fdc_id: long (nullable = true)
 |-- food_description: string (nullable = true)
 |-- food_type: string (nullable = true)
 |-- total_nutrients: long (nullable = true)
 |-- energy: double (nullable = true)
 |-- protein: double (nullable = true)
 |-- carbs: double (nullable = true)
 |-- total_fat: double (nullable = true)
 |-- water: double (nullable = true)
 |-- ash: double (nullable = true)
 |-- alcohol: double (nullable = true)
 |-- caffeine: double (nullable = true)
 |-- fiber: double (nullable = true)
 |-- sugars: double (nullable = true)
 |-- glucose: double (nullable = true)
 |-- fructose: double (nullable = true)
 |-- sucrose: double (nullable = true)
 |-- lactose: double (nullable = true)
 |-- saturated_fat: double (nullable = true)
 |-- monounsaturated_fat: double (nullable = true)
 |-- polyunsaturated_fat: double (nullable = true)
 |-- trans_fat: double (nullable = true)
 |-- cholesterol: double (nullable = true)
 |-- vitamin_a: double (null

nutritionalProfilesDF: org.apache.spark.sql.package.DataFrame = [fdc_id: bigint, food_description: string ... 46 more fields]
recipesDF: org.apache.spark.sql.package.DataFrame = [title: string, ingredients: string ... 5 more fields]

In [5]:
// Define allergen categories
val allergenCategories = Map(
    "eggs" -> eggAllergyMenus,
    "gluten" -> glutenFreeMenus,
    "lactose" -> lactoseIntolerantMenus,
    "nuts" -> nutAllergyMenus,
    "shellfish" -> shellfishAllergyMenus
)

// Days of the week
val daysOfWeek = Array("Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday")

println(s"Allergen categories: ${allergenCategories.keys.mkString(", ")}")
println(s"Days of week: ${daysOfWeek.mkString(", ")}")

Allergen categories: lactose, eggs, gluten, nuts, shellfish
Days of week: Monday, Tuesday, Wednesday, Thursday, Friday, Saturday, Sunday


allergenCategories: Map[String, org.apache.spark.sql.package.DataFrame] = Map(
  "lactose" -> [meal_id: bigint, starter_title: string ... 13 more fields],
  "eggs" -> [meal_id: bigint, starter_title: string ... 13 more fields],
  "gluten" -> [meal_id: bigint, starter_title: string ... 13 more fields],
  "nuts" -> [meal_id: bigint, starter_title: string ... 13 more fields],
  "shellfish" -> [meal_id: bigint, starter_title: string ... 13 more fields]
)
daysOfWeek: Array[String] = Array(
  "Monday",
  "Tuesday",
  "Wednesday",
  "Thursday",
  "Friday",
  "Saturday",
  "Sunday"
)

In [6]:
// Function to prepare features for linear regression
// NOTE: If you see errors about 'rating' column, restart the kernel and re-run all cells
def prepareFeatures(df: org.apache.spark.sql.DataFrame): org.apache.spark.sql.DataFrame = {
    // Add day of week feature (if not present)
    val dfWithDay = if (df.columns.contains("day_of_week")) {
        df
    } else {
        df.withColumn("day_of_week", 
            when(rand() < 0.143, "Monday")
            .when(rand() < 0.286, "Tuesday")
            .when(rand() < 0.429, "Wednesday")
            .when(rand() < 0.572, "Thursday")
            .when(rand() < 0.715, "Friday")
            .when(rand() < 0.858, "Saturday")
            .otherwise("Sunday"))
    }
    
    // Create a preference score (target variable for regression)
    // This could be based on ratings, nutritional value, or other metrics
    val dfWithScore = if (df.columns.contains("preference_score")) {
        dfWithDay
    } else {
        // Create a synthetic preference score based on available features
        // Since 'rating' column doesn't exist, use a base score with randomness
        // You can modify this to use actual features from your data (e.g., meal_id, ingredients count, etc.)
        dfWithDay.withColumn("preference_score", 
            lit(3.5) + rand() * 2.0 - 1.0) // Base score of 3.5 with randomness between 2.5 and 4.5
    }
    
    dfWithScore
}

defined function prepareFeatures

In [7]:
// Function to train linear regression model and generate recommendations
def trainModelAndRecommend(
    allergenType: String,
    menuDF: org.apache.spark.sql.DataFrame,
    dayOfWeek: String
): org.apache.spark.sql.DataFrame = {
    
    println(s"\n=== Processing: $allergenType - $dayOfWeek ===")
    
    // Prepare features
    val preparedDF = prepareFeatures(menuDF)
    
    // Filter for the specific day if day_of_week column exists
    val dayDF = if (preparedDF.columns.contains("day_of_week")) {
        preparedDF.filter(col("day_of_week") === dayOfWeek)
    } else {
        preparedDF // Use all data if day filtering not available
    }
    
    if (dayDF.count() == 0) {
        println(s"No data available for $allergenType on $dayOfWeek")
        return spark.emptyDataFrame
    }
    
    // Identify feature columns (exclude target and metadata columns)
    val excludeCols = Array("preference_score", "day_of_week", "menu_id", "restaurant_id", "menu_name", "restaurant_name")
    val featureCols = dayDF.columns.filterNot(excludeCols.contains)
    
    // Select numeric columns for features only (VectorAssembler doesn't support StringType)
    val numericCols = featureCols.filter { colName =>
        val dtype = dayDF.schema(colName).dataType
        dtype.isInstanceOf[NumericType]  // Only numeric types, exclude StringType
    }
    
    if (numericCols.length == 0) {
        println(s"No suitable feature columns found for $allergenType")
        return dayDF.limit(10) // Return top 10 as fallback
    }
    
    // Create feature vector
    val assembler = new VectorAssembler()
        .setInputCols(numericCols.take(10)) // Limit to first 10 numeric columns
        .setOutputCol("features")
        .setHandleInvalid("skip")
    
    // Select columns properly - convert all to Column objects
    // Exclude preference_score from dayDF.columns since we're adding it explicitly
    val otherCols = dayDF.columns.filterNot(_ == "preference_score").map(col)
    val allCols = Seq(col("features"), col("preference_score")) ++ otherCols
    val featureDF = assembler.transform(dayDF)
        .select(allCols: _*)
        .filter(col("features").isNotNull)
    
    if (featureDF.count() == 0) {
        println(s"No valid features for $allergenType on $dayOfWeek")
        return dayDF.limit(10)
    }
    
    // Train Linear Regression model
    val lr = new LinearRegression()
        .setLabelCol("preference_score")
        .setFeaturesCol("features")
        .setMaxIter(10)
        .setRegParam(0.3)
        .setElasticNetParam(0.8)
    
    val model = lr.fit(featureDF)
    
    // Make predictions
    val predictions = model.transform(featureDF)
        .withColumn("allergen_type", lit(allergenType))
        .withColumn("recommended_day", lit(dayOfWeek))
    
    // Get top recommendations (highest predicted scores)
    // Build column list properly - convert all to Column objects
    val recOtherCols = dayDF.columns.filterNot(Array("features", "preference_score").contains).map(col)
    val recCols = Seq(col("allergen_type"), col("recommended_day"), col("prediction")) ++ recOtherCols
    val recommendations = predictions
        .orderBy(desc("prediction"))
        .limit(5)
        .select(recCols: _*)
    
    println(s"Generated ${recommendations.count()} recommendations for $allergenType on $dayOfWeek")
    
    recommendations
}

defined function trainModelAndRecommend

In [8]:
// Generate recommendations for all allergen categories and days
import scala.collection.mutable.ListBuffer

val allRecommendations = ListBuffer[org.apache.spark.sql.DataFrame]()

for ((allergenType, menuDF) <- allergenCategories) {
    for (day <- daysOfWeek) {
        val recommendations = trainModelAndRecommend(allergenType, menuDF, day)
        if (recommendations.count() > 0) {
            allRecommendations += recommendations
        }
    }
}

// Combine all recommendations
val finalRecommendations = if (allRecommendations.nonEmpty) {
    allRecommendations.reduce(_ union _)
} else {
    spark.emptyDataFrame
}

println(s"\n=== Total Recommendations Generated: ${finalRecommendations.count()} ===")


=== Processing: lactose - Monday ===
No data available for lactose on Monday

=== Processing: lactose - Tuesday ===
Generated 5 recommendations for lactose on Tuesday

=== Processing: lactose - Wednesday ===
Generated 4 recommendations for lactose on Wednesday

=== Processing: lactose - Thursday ===
Generated 3 recommendations for lactose on Thursday

=== Processing: lactose - Friday ===
Generated 2 recommendations for lactose on Friday

=== Processing: lactose - Saturday ===
Generated 1 recommendations for lactose on Saturday

=== Processing: lactose - Sunday ===
No data available for lactose on Sunday

=== Processing: eggs - Monday ===
Generated 1 recommendations for eggs on Monday

=== Processing: eggs - Tuesday ===
Generated 5 recommendations for eggs on Tuesday

=== Processing: eggs - Wednesday ===
Generated 5 recommendations for eggs on Wednesday

=== Processing: eggs - Thursday ===
Generated 3 recommendations for eggs on Thursday

=== Processing: eggs - Friday ===
Generated 2 r

import scala.collection.mutable.ListBuffer


allRecommendations: ListBuffer[org.apache.spark.sql.package.DataFrame] = ListBuffer(
  [allergen_type: string, recommended_day: string ... 17 more fields],
  [allergen_type: string, recommended_day: string ... 17 more fields],
  [allergen_type: string, recommended_day: string ... 17 more fields],
  [allergen_type: string, recommended_day: string ... 17 more fields],
  [allergen_type: string, recommended_day: string ... 17 more fields],
  [allergen_type: string, recommended_day: string ... 17 more fields],
  [allergen_type: string, recommended_day: string ... 17 more fields],
  [allergen_type: string, recommended_day: string ... 17 more fields],
  [allergen_type: string, recommended_day: string ... 17 more fields],
  [allergen_type: string, recommended_day: string ... 17 more fields],
  [allergen_type: string, recommended_day: string ... 17 more fields],
  [allergen_type: string, recommended_day: string ... 17 more fields],
  [allergen_type: 

In [9]:
// Display recommendations grouped by allergen type and day
for ((allergenType, _) <- allergenCategories) {
    println(s"\n{'='*60}")
    println(s"RECOMMENDATIONS FOR ${allergenType.toUpperCase} ALLERGY")
    println(s"{'='*60}")
    
    for (day <- daysOfWeek) {
        val dayRecs = finalRecommendations
            .filter(col("allergen_type") === allergenType && col("recommended_day") === day)
            .orderBy(desc("prediction"))
        
        println(s"\n--- $day ---")
        dayRecs.show(5, truncate=false)
    }
}


{'='*60}
RECOMMENDATIONS FOR LACTOSE ALLERGY
{'='*60}

--- Monday ---
+-------------+---------------+----------+-------+-------------+-------------------+------------------+------------+----------+----------------+---------------+---------+-------------+-------------------+------------------+------------+-------------------+-----------------------+-----------+
|allergen_type|recommended_day|prediction|meal_id|starter_title|starter_ingredients|starter_directions|starter_link|main_title|main_ingredients|main_directions|main_link|dessert_title|dessert_ingredients|dessert_directions|dessert_link|dietary_restriction|restriction_description|day_of_week|
+-------------+---------------+----------+-------+-------------+-------------------+------------------+------------+----------+----------------+---------------+---------+-------------+-------------------+------------------+------------+-------------------+-----------------------+-----------+
+-------------+---------------+----------+-------+

In [10]:
// Save recommendations to parquet
val outputPath = "output/menu_recommendations_scala"
finalRecommendations
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .parquet(outputPath)

println(s"Recommendations saved to: $outputPath")

// Convert array columns to strings for CSV export (CSV doesn't support ARRAY types)
val csvDF = finalRecommendations.columns.foldLeft(finalRecommendations) { (df, colName) =>
    val dtype = df.schema(colName).dataType
    if (dtype.isInstanceOf[ArrayType]) {
        // Convert array to comma-separated string
        df.withColumn(colName, concat_ws(", ", col(colName)))
    } else {
        df
    }
}

// Save as CSV for easier viewing
csvDF
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(s"$outputPath/csv")

println(s"CSV version saved to: $outputPath/csv")

Recommendations saved to: output/menu_recommendations_scala
CSV version saved to: output/menu_recommendations_scala/csv


outputPath: String = "output/menu_recommendations_scala"
csvDF: org.apache.spark.sql.package.DataFrame = [allergen_type: string, recommended_day: string ... 17 more fields]

In [11]:
// Summary statistics
val summaryDF = finalRecommendations
    .groupBy("allergen_type", "recommended_day")
    .agg(
        count("*").alias("num_recommendations"),
        avg("prediction").alias("avg_prediction_score"),
        max("prediction").alias("max_prediction_score"),
        min("prediction").alias("min_prediction_score")
    )
    .orderBy("allergen_type", "recommended_day")

println("\n=== Recommendation Summary Statistics ===")
summaryDF.show(35, truncate=false)


=== Recommendation Summary Statistics ===
+-------------+---------------+-------------------+--------------------+--------------------+--------------------+
|allergen_type|recommended_day|num_recommendations|avg_prediction_score|max_prediction_score|min_prediction_score|
+-------------+---------------+-------------------+--------------------+--------------------+--------------------+
|eggs         |Friday         |2                  |3.8768270010836847  |4.002966234752153   |3.7506877674152164  |
|eggs         |Monday         |1                  |2.9406316081116817  |2.9406316081116817  |2.9406316081116817  |
|eggs         |Saturday       |1                  |2.533670531696497   |2.533670531696497   |2.533670531696497   |
|eggs         |Thursday       |3                  |3.0279552378161596  |3.0279552378161596  |3.0279552378161596  |
|eggs         |Tuesday        |5                  |3.6332855425197237  |3.6332855425197237  |3.6332855425197237  |
|eggs         |Wednesday      |5     

summaryDF: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [allergen_type: string, recommended_day: string ... 4 more fields]

In [11]:
// Note: Spark session cleanup moved to end of notebook (after ALS section)

# ALS Collaborative Filtering Recommender System

This section implements an ALS (Alternating Least Squares) collaborative filtering recommender system based on:
- **Users-Food Allergy data**: 10,002 users with allergy categories
- **Recipes data**: 2.23 million recipes

The system will recommend recipes to users based on their allergy restrictions and collaborative filtering patterns.

In [12]:
// Import ALS and additional Spark MLlib components
import org.apache.spark.ml.recommendation.ALS
import org.apache.spark.ml.evaluation.RegressionEvaluator
import org.apache.spark.sql.expressions.Window

println("ALS and evaluation components imported!")

ALS and evaluation components imported!


import org.apache.spark.ml.recommendation.ALS

import org.apache.spark.ml.evaluation.RegressionEvaluator

import org.apache.spark.sql.expressions.Window



In [13]:
// Load users-food allergy data
val usersAllergyDF = spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("data/10_002_users-food_allergy.csv")

println("Users-food allergy data loaded!")
println(s"Total users: ${usersAllergyDF.count()}")
usersAllergyDF.printSchema()
usersAllergyDF.show(10, truncate=false)

Users-food allergy data loaded!
Total users: 10000
root
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Family_History: string (nullable = true)
 |-- Previous_Reaction: string (nullable = true)
 |-- Symptoms: string (nullable = true)
 |-- Food_Type: string (nullable = true)
 |-- Food_Frequency: integer (nullable = true)
 |-- Medical_Conditions: string (nullable = true)
 |-- IgE_Levels: double (nullable = true)
 |-- Severity_Score: integer (nullable = true)
 |-- Allergic: integer (nullable = true)

+---+------+--------------+-----------------+----------------+---------+--------------+------------------+------------------+--------------+--------+
|Age|Gender|Family_History|Previous_Reaction|Symptoms        |Food_Type|Food_Frequency|Medical_Conditions|IgE_Levels        |Severity_Score|Allergic|
+---+------+--------------+-----------------+----------------+---------+--------------+------------------+------------------+--------------+--------+
|56 |Other |No  

usersAllergyDF: org.apache.spark.sql.package.DataFrame = [Age: int, Gender: string ... 9 more fields]

In [14]:
// Load recipes data
val recipesDF = spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("data/2_231_151_recipes_data.csv")

println("Recipes data loaded!")
println(s"Total recipes: ${recipesDF.count()}")
recipesDF.printSchema()
recipesDF.show(5, truncate=false)

Recipes data loaded!
Total recipes: 2231149
root
 |-- title: string (nullable = true)
 |-- ingredients: string (nullable = true)
 |-- directions: string (nullable = true)
 |-- link: string (nullable = true)
 |-- source: string (nullable = true)
 |-- NER: string (nullable = true)
 |-- site: string (nullable = true)

+---------------------+------------------------------------+--------------------------------------+----------------------------------------------+-------------------------------------+-----------------------------------+----------------------------------------------------------------------------+
|title                |ingredients                         |directions                            |link                                          |source                               |NER                                |site                                                                        |
+---------------------+------------------------------------+---------------------------

recipesDF: org.apache.spark.sql.package.DataFrame = [title: string, ingredients: string ... 5 more fields]

In [15]:
// Explore allergy distribution in users data
println("=== Allergy Category Distribution ===")
val allergyCols = usersAllergyDF.columns.filter { colName =>
    val lower = colName.toLowerCase
    lower.contains("allergy") || 
    lower.contains("eggs") ||
    lower.contains("gluten") ||
    lower.contains("lactose") ||
    lower.contains("nuts") ||
    lower.contains("shellfish")
}

if (allergyCols.nonEmpty) {
    allergyCols.foreach { colName =>
        val count = usersAllergyDF.filter(col(colName) === 1 || col(colName) === true || col(colName) === "yes").count()
        println(s"$colName: $count users")
    }
} else {
    println("Available columns:")
    usersAllergyDF.columns.foreach(println)
    println("\nFirst few rows:")
    usersAllergyDF.show(20, truncate=false)
}

=== Allergy Category Distribution ===
Available columns:
Age
Gender
Family_History
Previous_Reaction
Symptoms
Food_Type
Food_Frequency
Medical_Conditions
IgE_Levels
Severity_Score
Allergic

First few rows:
+---+------+--------------+-----------------+----------------+---------+--------------+------------------+------------------+--------------+--------+
|Age|Gender|Family_History|Previous_Reaction|Symptoms        |Food_Type|Food_Frequency|Medical_Conditions|IgE_Levels        |Severity_Score|Allergic|
+---+------+--------------+-----------------+----------------+---------+--------------+------------------+------------------+--------------+--------+
|56 |Other |No            |Moderate         |Swelling        |Gluten   |15            |Asthma            |581.122867931843  |8             |1       |
|19 |Male  |No            |None             |Swelling        |Eggs     |12            |None              |79.47676746934107 |9             |1       |
|76 |Male  |Yes           |Severe           

allergyCols: Array[String] = Array()

In [16]:
// Prepare data for ALS: Create user-recipe interactions
// We'll create synthetic ratings/interactions based on allergy compatibility
// This is a simplified approach - in production, you'd use actual user ratings

// Step 1: Identify user ID and recipe ID columns
val userIdCol = usersAllergyDF.columns.find { colName =>
    val lower = colName.toLowerCase
    lower.contains("user") || lower.contains("id")
}.getOrElse(usersAllergyDF.columns(0))

val recipeIdCol = recipesDF.columns.find { colName =>
    val lower = colName.toLowerCase
    lower.contains("recipe") && lower.contains("id")
}.getOrElse(recipesDF.columns(0))

println(s"Using user ID column: $userIdCol")
println(s"Using recipe ID column: $recipeIdCol")

// Step 2: Create numeric IDs for ALS (ALS requires numeric user and item IDs)
import org.apache.spark.ml.feature.StringIndexer

val userIndexer = new StringIndexer()
    .setInputCol(userIdCol)
    .setOutputCol("user_id_numeric")
    .fit(usersAllergyDF)

val recipeIndexer = new StringIndexer()
    .setInputCol(recipeIdCol)
    .setOutputCol("recipe_id_numeric")
    .fit(recipesDF)

val usersIndexed = userIndexer.transform(usersAllergyDF)
val recipesIndexed = recipeIndexer.transform(recipesDF)

// Sample users to reduce memory pressure (use 30% of users)
val sampledUsersIndexed = usersIndexed.sample(false, 0.3, seed=42).cache()

println(s"Indexed ${usersIndexed.count()} total users")
println(s"Using ${sampledUsersIndexed.count()} sampled users (30%) for ALS training")
println(s"Indexed ${recipesIndexed.count()} recipes")

Using user ID column: Age
Using recipe ID column: title
Indexed 10000 total users
Using 3089 sampled users (30%) for ALS training
Indexed 2231149 recipes


userIdCol: String = "Age"
recipeIdCol: String = "title"
import org.apache.spark.ml.feature.StringIndexer


userIndexer: org.apache.spark.ml.feature.StringIndexerModel = StringIndexerModel: uid=strIdx_0b8c440a7b01, handleInvalid=error
recipeIndexer: org.apache.spark.ml.feature.StringIndexerModel = StringIndexerModel: uid=strIdx_5d6c395bc417, handleInvalid=error
usersIndexed: org.apache.spark.sql.package.DataFrame = [Age: int, Gender: string ... 10 more fields]
recipesIndexed: org.apache.spark.sql.package.DataFrame = [title: string, ingredients: string ... 6 more fields]
sampledUsersIndexed: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [Age: int, Gender: string ... 10 more fields]

In [17]:
// Create user-recipe interaction matrix based on allergy compatibility
// For each user, assign ratings to recipes based on allergy restrictions
// Higher ratings for recipes that are compatible with user's allergies

// Sample recipes once (for performance, limit to subset)
// Aggressively reduced sampling rate to 0.1% to reduce memory pressure
val sampledRecipes = recipesIndexed
    .sample(false, 0.001, seed=42) // Sample 0.1% of recipes (reduced from 0.5%)
    .select("recipe_id_numeric")
    .distinct()
    .cache()

println(s"Sampled ${sampledRecipes.count()} recipes for interaction matrix (0.1% of total)")

// Identify allergy-related columns based on actual data structure
// Look for: symptoms, IgE_Levels, Severity_Score, and Allergic columns
val symptomsCol = usersIndexed.columns.find { colName =>
    val lower = colName.toLowerCase
    lower.contains("symptom")
}.getOrElse(null)

val igeLevelsCol = usersIndexed.columns.find { colName =>
    val lower = colName.toLowerCase
    lower.contains("ige") || lower.contains("ige_level")
}.getOrElse(null)

val severityScoreCol = usersIndexed.columns.find { colName =>
    val lower = colName.toLowerCase
    lower.contains("severity") || lower.contains("severity_score")
}.getOrElse(null)

val allergicCol = usersIndexed.columns.find { colName =>
    val lower = colName.toLowerCase
    lower.contains("allergic")
}.getOrElse(null)

println(s"Found columns:")
println(s"  Symptoms: ${if (symptomsCol != null) symptomsCol else "Not found"}")
println(s"  IgE_Levels: ${if (igeLevelsCol != null) igeLevelsCol else "Not found"}")
println(s"  Severity_Score: ${if (severityScoreCol != null) severityScoreCol else "Not found"}")
println(s"  Allergic: ${if (allergicCol != null) allergicCol else "Not found"}")

// Collect recipe IDs ONCE before the RDD transformation (not inside it)
val recipeIdsArray = sampledRecipes.collect().map(_.getAs[Double]("recipe_id_numeric"))
println(s"Collected ${recipeIdsArray.length} recipe IDs for interaction matrix")

// Broadcast recipe IDs for better performance
import org.apache.spark.broadcast.Broadcast
val broadcastRecipeIds = spark.sparkContext.broadcast(recipeIdsArray)

// Create interactions: For each user, assign ratings to sampled recipes
// Determine if user has allergies based on Allergic column (value 1) and other indicators
import spark.implicits._

// Select columns needed for allergy detection
val columnsToSelect = Seq("user_id_numeric") ++ 
    Seq(symptomsCol, igeLevelsCol, severityScoreCol, allergicCol).filter(_ != null)

val interactionsDF = sampledUsersIndexed  // Use sampled users instead of all users
    .select(columnsToSelect.map(col): _*)
    .rdd.flatMap { row =>
        val userId = row.getAs[Double]("user_id_numeric")
        
        // Determine if user has allergies based on:
        // 1. Allergic column = 1 (primary indicator)
        // 2. Presence of symptoms
        // 3. IgE_Levels > 0 or above threshold
        // 4. Severity_Score > 0 or above threshold
        
        val isAllergic = if (allergicCol != null) {
            val allergicValue = row.getAs[Any](allergicCol)
            allergicValue != null && (allergicValue == 1 || allergicValue == true || allergicValue == "1" || allergicValue == "Yes")
        } else {
            false
        }
        
        val hasSymptoms = if (symptomsCol != null) {
            val symptomsValue = row.getAs[Any](symptomsCol)
            symptomsValue != null && symptomsValue.toString.trim.nonEmpty && symptomsValue.toString.toLowerCase != "none"
        } else {
            false
        }
        
        val hasElevatedIgE = if (igeLevelsCol != null) {
            try {
                val igeValue = row.getAs[Any](igeLevelsCol)
                if (igeValue != null) {
                    val igeNum = igeValue match {
                        case n: Number => n.doubleValue()
                        case s: String => s.toDouble
                        case _ => 0.0
                    }
                    igeNum > 0.0 // Consider > 0 as elevated
                } else {
                    false
                }
            } catch {
                case _: Exception => false
            }
        } else {
            false
        }
        
        val hasSeverity = if (severityScoreCol != null) {
            try {
                val severityValue = row.getAs[Any](severityScoreCol)
                if (severityValue != null) {
                    val severityNum = severityValue match {
                        case n: Number => n.doubleValue()
                        case s: String => s.toDouble
                        case _ => 0.0
                    }
                    severityNum > 0.0 // Consider > 0 as having severity
                } else {
                    false
                }
            } catch {
                case _: Exception => false
            }
        } else {
            false
        }
        
        // User is considered allergic if Allergic=1 OR (has symptoms AND (elevated IgE OR severity score))
        val userHasAllergies = isAllergic || (hasSymptoms && (hasElevatedIgE || hasSeverity))
        
        // Use broadcast recipe IDs (already collected)
        broadcastRecipeIds.value.map { recipeId =>
            // Base rating: 3.0 (neutral)
            // Adjust rating based on allergy status
            // Users with allergies get slightly lower base ratings (they need compatible recipes)
            // Users without allergies get higher base ratings (more flexibility)
            var rating = if (userHasAllergies) {
                2.5 + scala.util.Random.nextDouble() * 2.0 // 2.5-4.5 for allergic users
            } else {
                3.5 + scala.util.Random.nextDouble() * 1.5 // 3.5-5.0 for non-allergic users
            }
            
            (userId.toInt, recipeId.toInt, rating)
        }
    }
    .toDF("user_id", "recipe_id", "rating")

println(s"Created ${interactionsDF.count()} user-recipe interactions")
println("\nSample interactions:")
interactionsDF.show(10)

Sampled 2227 recipes for interaction matrix (0.1% of total)
Found columns:
  Symptoms: Symptoms
  IgE_Levels: IgE_Levels
  Severity_Score: Severity_Score
  Allergic: Allergic
Collected 2227 recipe IDs for interaction matrix
Created 6879203 user-recipe interactions

Sample interactions:
+-------+---------+------------------+
|user_id|recipe_id|            rating|
+-------+---------+------------------+
|     27|      147|2.7355405858152926|
|     27|    26617| 3.759747663861616|
|     27|      169|2.7644692908128015|
|     27|     2317| 4.446575369386192|
|     27|       67| 3.216036800841513|
|     27|        0|3.1803814789311957|
|     27|   141864|3.7826539249640527|
|     27|     2853|3.5024991870953848|
|     27|    77441| 4.220093396746935|
|     27|   555457|3.4000948052632065|
+-------+---------+------------------+
only showing top 10 rows



sampledRecipes: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [recipe_id_numeric: double]
symptomsCol: String = "Symptoms"
igeLevelsCol: String = "IgE_Levels"
severityScoreCol: String = "Severity_Score"
allergicCol: String = "Allergic"
recipeIdsArray: Array[Double] = Array(
  147.0,
  26617.0,
  169.0,
  2317.0,
  67.0,
  0.0,
  141864.0,
  2853.0,
  77441.0,
  555457.0,
  303649.0,
  3117.0,
  157565.0,
  827.0,
  521.0,
  124.0,
  59758.0,
  675204.0,
  13089.0,
  838367.0,
  817004.0,
  410.0,
  201.0,
  2483.0,
  167032.0,
  10483.0,
  1060.0,
  11472.0,
  17623.0,
  406660.0,
  621910.0,
  1611.0,
  2771.0,
  566161.0,
  131474.0,
  181.0,
  56292.0,
  733.0,
...
import org.apache.spark.broadcast.Broadcast

broadcastRecipeIds: Broadcast[Array[Double]] = Broadcast(657)
import spark.implicits._

// Select columns needed for allergy detection

columnsToSelect: Seq[String] = List(
  "user_id_numeric",
  "Symptoms",
  "IgE_Levels",
  "Severity_Score",
  "Allergic"
)
interact

In [18]:
// Split data into training and test sets
// Cache and checkpoint the interactions DF to help with memory
val interactionsCached = interactionsDF.cache()
interactionsCached.checkpoint()  // Checkpoint to disk to free memory

val Array(training, test) = interactionsCached.randomSplit(Array(0.8, 0.2), seed=42)

// Cache training set and checkpoint it
val trainingCached = training.cache()
trainingCached.checkpoint()

println(s"Training set: ${trainingCached.count()} interactions")
println(s"Test set: ${test.count()} interactions")
println("Data checkpointed to disk to free memory")

Training set: 5503841 interactions
Test set: 1375362 interactions
Data checkpointed to disk to free memory


interactionsCached: org.apache.spark.sql.package.DataFrame = [user_id: int, recipe_id: int ... 1 more field]
res17_1: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [user_id: int, recipe_id: int ... 1 more field]
training: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [user_id: int, recipe_id: int ... 1 more field]
test: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [user_id: int, recipe_id: int ... 1 more field]
trainingCached: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [user_id: int, recipe_id: int ... 1 more field]
res17_4: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [user_id: int, recipe_id: int ... 1 more field]

In [19]:
// Build ALS recommendation model with memory-optimized parameters
val als = new ALS()
    .setMaxIter(3)  // Further reduced from 5 to 3 to minimize memory usage
    .setRegParam(0.1)
    .setRank(10)  // Further reduced from 20 to 10 to significantly reduce memory usage
    .setUserCol("user_id")
    .setItemCol("recipe_id")
    .setRatingCol("rating")
    .setColdStartStrategy("drop")  // Drop users/items not in training set
    .setNonnegative(true)
    .setImplicitPrefs(false)  // Explicit feedback
    .setAlpha(1.0)  // Only used for implicit feedback
    .setNumUserBlocks(200)  // Increase blocks for better parallelism
    .setNumItemBlocks(200)  // Increase blocks for better parallelism

println("Training ALS model...")
println("Note: Using reduced rank (10) and iterations (3) to manage memory")
println("Note: Using sampled users (30%) and recipes (0.1%) to reduce dataset size")
println("Expected interactions: ~3,000 users × ~2,200 recipes = ~6.6M interactions")
val model = als.fit(trainingCached)  // Use cached training set
println("ALS model trained successfully!")

Training ALS model...
Note: Using reduced rank (10) and iterations (3) to manage memory
Note: Using sampled users (30%) and recipes (0.1%) to reduce dataset size
Expected interactions: ~3,000 users × ~2,200 recipes = ~6.6M interactions
ALS model trained successfully!


als: ALS = als_85bd19eec84c
model: org.apache.spark.ml.recommendation.ALSModel = ALSModel: uid=als_85bd19eec84c, rank=10

In [20]:
// Evaluate the model
val predictions = model.transform(test)
val evaluator = new RegressionEvaluator()
    .setMetricName("rmse")
    .setLabelCol("rating")
    .setPredictionCol("prediction")

val rmse = evaluator.evaluate(predictions)
println(s"Root Mean Squared Error (RMSE) = $rmse")

// Show sample predictions
predictions.filter(col("prediction").isNotNull)
    .orderBy(desc("prediction"))
    .show(10, truncate=false)

Root Mean Squared Error (RMSE) = 0.5841898495174899
+-------+---------+------------------+----------+
|user_id|recipe_id|rating            |prediction|
+-------+---------+------------------+----------+
|5      |1135597  |2.7029726468127886|3.4534132 |
|5      |1135597  |2.680278718259715 |3.4534132 |
|5      |1135597  |2.70720009246581  |3.4534132 |
|5      |1135597  |2.759705670284344 |3.4534132 |
|5      |1135597  |2.7799542190228546|3.4534132 |
|5      |1135597  |3.737361416232071 |3.4534132 |
|5      |1135597  |3.8417347415985343|3.4534132 |
|5      |1135597  |4.433348144997025 |3.4534132 |
|58     |1135597  |4.1267082220092375|3.450311  |
|58     |1135597  |3.373428986905307 |3.450311  |
+-------+---------+------------------+----------+
only showing top 10 rows



predictions: org.apache.spark.sql.package.DataFrame = [user_id: int, recipe_id: int ... 2 more fields]
evaluator: RegressionEvaluator = RegressionEvaluator: uid=regEval_2d39d887a63d, metricName=rmse, throughOrigin=false
rmse: Double = 0.5841898495174899

In [21]:
// Generate top N recommendations for each user
val userRecs = model.recommendForAllUsers(10) // Top 10 recommendations per user

println(s"Generated recommendations for ${userRecs.count()} users")
userRecs.show(5, truncate=false)

// Flatten recommendations for easier analysis
val userRecsFlat = userRecs
    .select(col("user_id"), explode(col("recommendations")).as("rec"))
    .select("user_id", "rec.recipe_id", "rec.rating")
    .withColumnRenamed("rating", "predicted_rating")

println("Flattened recommendations:")
userRecsFlat.show(20, truncate=false)

Generated recommendations for 75 users
+-------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|user_id|recommendations                                                                                                                                                                                                 |
+-------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|1      |[{1135597, 3.444409}, {18987, 3.4407537}, {662844, 3.4405882}, {126848, 3.4401252}, {224763, 3.4394517}, {1078582, 3.4390671}, {725877, 3.438251}, {1308883, 3.4374554}, {39712, 3.437346}, {6843, 3.4373367}]  |
|3      |[{1135597, 3.4465864}, {18987, 3.442929}, {662844, 3.4427633}, {126848, 3.44

: 

In [21]:
// Join with original user and recipe data to get details
// NOTE: Make sure you've run cells 2, 18, and 23 before running this cell
// Cell 2: Imports Spark functions (including col)
// Cell 18: Creates usersIndexed and recipesIndexed
// Cell 23: Creates userRecsFlat

import org.apache.spark.sql.functions._

val recommendationsWithDetails = userRecsFlat
    .join(usersIndexed.select("user_id_numeric", usersIndexed.columns.filterNot(_ == "user_id_numeric"): _*), 
          col("user_id") === col("user_id_numeric"), "left")
    .join(recipesIndexed.select("recipe_id_numeric", recipesIndexed.columns.filterNot(_ == "recipe_id_numeric"): _*),
          col("recipe_id") === col("recipe_id_numeric"), "left")
    .select((Seq("user_id", "recipe_id", "predicted_rating") ++ 
             usersIndexed.columns.filterNot(_ == "user_id_numeric").take(5) ++
             recipesIndexed.columns.filterNot(_ == "recipe_id_numeric").take(5)): _*)

println("Recommendations with user and recipe details:")
recommendationsWithDetails.show(10, truncate=false)

cmd21.sc:3: not found: value userRecsFlat
val recommendationsWithDetails = userRecsFlat
                                 ^Compilation Failed

: 

# Visualizations

This section creates visual representations of the recommender system outputs using HTML/JavaScript charts and Spark SQL aggregations.

In [21]:
// Prepare data for visualization: Statistics and aggregations
import org.apache.spark.sql.functions._

// 1. Recommendation distribution by predicted rating
val ratingDistribution = userRecsFlat
    .select(
        when(col("predicted_rating") >= 4.5, "4.5-5.0")
        .when(col("predicted_rating") >= 4.0, "4.0-4.5")
        .when(col("predicted_rating") >= 3.5, "3.5-4.0")
        .when(col("predicted_rating") >= 3.0, "3.0-3.5")
        .otherwise("Below 3.0")
        .as("rating_range"),
        col("predicted_rating")
    )
    .groupBy("rating_range")
    .agg(
        count("*").as("count"),
        avg("predicted_rating").as("avg_rating")
    )
    .orderBy("rating_range")

println("=== Rating Distribution ===")
ratingDistribution.show(false)

cmd21.sc:4: not found: value userRecsFlat
val ratingDistribution = userRecsFlat
                         ^Compilation Failed

: 

In [21]:
// 2. Top recommended recipes (most frequently recommended)
val topRecipes = userRecsFlat
    .groupBy("recipe_id")
    .agg(
        count("*").as("recommendation_count"),
        avg("predicted_rating").as("avg_predicted_rating"),
        max("predicted_rating").as("max_predicted_rating")
    )
    .orderBy(desc("recommendation_count"))
    .limit(20)

println("=== Top 20 Most Recommended Recipes ===")
topRecipes.show(false)

cmd21.sc:1: not found: value userRecsFlat
val topRecipes = userRecsFlat
                 ^Compilation Failed

: 

In [ ]:
// 3. Users with most recommendations (active users)
val activeUsers = userRecsFlat
    .groupBy("user_id")
    .agg(
        count("*").as("recommendation_count"),
        avg("predicted_rating").as("avg_predicted_rating")
    )
    .orderBy(desc("recommendation_count"))
    .limit(20)

println("=== Top 20 Most Active Users ===")
activeUsers.show(false)

In [ ]:
// 4. Create HTML visualization using Scala string interpolation
// This creates an HTML page with charts using Chart.js

def createVisualizationHTML(
    ratingDist: org.apache.spark.sql.DataFrame,
    topRecipes: org.apache.spark.sql.DataFrame,
    activeUsers: org.apache.spark.sql.DataFrame
): String = {
    
    // Collect data for visualization
    val ratingData = ratingDist.collect()
    val recipeData = topRecipes.limit(10).collect()
    val userData = activeUsers.limit(10).collect()
    
    val ratingLabels = ratingData.map(_.getAs[String]("rating_range"))
    val ratingCounts = ratingData.map(_.getAs[Long]("count"))
    
    val recipeIds = recipeData.map(_.getAs[Int]("recipe_id").toString)
    val recipeCounts = recipeData.map(_.getAs[Long]("recommendation_count"))
    
    val userIds = userData.map(_.getAs[Int]("user_id").toString)
    val userCounts = userData.map(_.getAs[Long]("recommendation_count"))
    
    s"""
    <!DOCTYPE html>
    <html>
    <head>
        <title>ALS Recommender System Visualizations</title>
        <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
        <style>
            body { font-family: Arial, sans-serif; margin: 20px; }
            .chart-container { width: 800px; height: 400px; margin: 20px 0; }
            h2 { color: #333; }
        </style>
    </head>
    <body>
        <h1>ALS Collaborative Filtering Recommender System - Visualizations</h1>
        
        <h2>1. Rating Distribution</h2>
        <div class="chart-container">
            <canvas id="ratingChart"></canvas>
        </div>
        
        <h2>2. Top Recommended Recipes</h2>
        <div class="chart-container">
            <canvas id="recipeChart"></canvas>
        </div>
        
        <h2>3. Most Active Users</h2>
        <div class="chart-container">
            <canvas id="userChart"></canvas>
        </div>
        
        <script>
            // Rating Distribution Chart
            new Chart(document.getElementById('ratingChart'), {
                type: 'bar',
                data: {
                    labels: [${ratingLabels.map(l => s"'$l'").mkString(", ")}],
                    datasets: [{
                        label: 'Number of Recommendations',
                        data: [${ratingCounts.mkString(", ")}],
                        backgroundColor: 'rgba(54, 162, 235, 0.6)',
                        borderColor: 'rgba(54, 162, 235, 1)',
                        borderWidth: 1
                    }]
                },
                options: {
                    responsive: true,
                    maintainAspectRatio: false,
                    scales: {
                        y: { beginAtZero: true }
                    }
                }
            });
            
            // Top Recipes Chart
            new Chart(document.getElementById('recipeChart'), {
                type: 'bar',
                data: {
                    labels: [${recipeIds.map(id => s"'Recipe $id'").mkString(", ")}],
                    datasets: [{
                        label: 'Recommendation Count',
                        data: [${recipeCounts.mkString(", ")}],
                        backgroundColor: 'rgba(255, 99, 132, 0.6)',
                        borderColor: 'rgba(255, 99, 132, 1)',
                        borderWidth: 1
                    }]
                },
                options: {
                    responsive: true,
                    maintainAspectRatio: false,
                    scales: {
                        y: { beginAtZero: true }
                    }
                }
            });
            
            // Active Users Chart
            new Chart(document.getElementById('userChart'), {
                type: 'line',
                data: {
                    labels: [${userIds.map(id => s"'User $id'").mkString(", ")}],
                    datasets: [{
                        label: 'Recommendations Received',
                        data: [${userCounts.mkString(", ")}],
                        backgroundColor: 'rgba(75, 192, 192, 0.2)',
                        borderColor: 'rgba(75, 192, 192, 1)',
                        borderWidth: 2,
                        fill: true
                    }]
                },
                options: {
                    responsive: true,
                    maintainAspectRatio: false,
                    scales: {
                        y: { beginAtZero: true }
                    }
                }
            });
        </script>
    </body>
    </html>
    """
}

val htmlContent = createVisualizationHTML(ratingDistribution, topRecipes, activeUsers)
println("HTML visualization created!")
println("To view: Save the HTML content to a file and open in a browser")

In [ ]:
// Save visualization HTML to file
import java.nio.file.{Files, Paths, StandardOpenOption}

val htmlPath = "output/als_recommender_visualizations.html"
Files.write(
    Paths.get(htmlPath),
    htmlContent.getBytes,
    StandardOpenOption.CREATE,
    StandardOpenOption.TRUNCATE_EXISTING
)

println(s"Visualization saved to: $htmlPath")
println("Open this file in a web browser to view the charts!")

In [ ]:
// Save recommendations and statistics to parquet
val alsOutputPath = "output/als_recommendations"
userRecsFlat
    .coalesce(1)
    .write
    .mode("overwrite")
    .parquet(s"$alsOutputPath/recommendations")

ratingDistribution
    .coalesce(1)
    .write
    .mode("overwrite")
    .parquet(s"$alsOutputPath/rating_distribution")

topRecipes
    .coalesce(1)
    .write
    .mode("overwrite")
    .parquet(s"$alsOutputPath/top_recipes")

activeUsers
    .coalesce(1)
    .write
    .mode("overwrite")
    .parquet(s"$alsOutputPath/active_users")

println(s"ALS recommendations and statistics saved to: $alsOutputPath")

In [ ]:
// Display HTML visualization inline (if Jupyter supports HTML output)
// This will show the charts directly in the notebook
import org.apache.spark.sql.DataFrame

def displayHTML(html: String): Unit = {
    // In Jupyter, this would typically use IPython.display.HTML
    // For Scala notebooks, we'll print instructions
    println("=" * 80)
    println("VISUALIZATION READY")
    println("=" * 80)
    println("The HTML visualization has been saved to: output/als_recommender_visualizations.html")
    println("Open this file in a web browser to view interactive charts showing:")
    println("  - Rating distribution")
    println("  - Top recommended recipes")
    println("  - Most active users")
    println("=" * 80)
}

displayHTML(htmlContent)

In [ ]:
// Cleanup: Stop Spark session
spark.stop()
println("Spark session stopped.")